# Semana 10: Pronosticar la demanda con regresión

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
df = pd.read_csv(BASE + "bicicletas_dia.csv", parse_dates=["dteday"])
df["temp_c"] = df["temp"] * 41; df["hum_pct"] = df["hum"] * 100; df["viento_kmh"] = df["windspeed"] * 67
y = df["cnt"]

## 1. La recta

In [ ]:
m1 = LinearRegression().fit(df[["temp_c"]], y)
p1 = m1.predict(df[["temp_c"]])
print("ŷ =", round(m1.intercept_, 1), "+", round(m1.coef_[0], 1), "· °C")
print("R²", round(r2_score(y, p1), 3), " MAE", round(mean_absolute_error(y, p1)))
print("A 25 °C:", round(m1.predict(pd.DataFrame({"temp_c": [25]}))[0]))

## 2. Regresión múltiple

In [ ]:
X = pd.get_dummies(df[["temp_c", "hum_pct", "viento_kmh", "yr", "workingday", "season", "weathersit"]],
                   columns=["season", "weathersit"], drop_first=True).astype(float)
m2 = LinearRegression().fit(X, y)
p2 = m2.predict(X)
print(pd.Series(m2.coef_, index=X.columns).round(1))
print("Intercepto", round(m2.intercept_, 1), " R²", round(r2_score(y, p2), 3), " MAE", round(mean_absolute_error(y, p2)))

## 3. Residuos

In [ ]:
df["residuo"] = y - p2
peores = df.loc[df["residuo"].abs().nlargest(5).index, ["dteday", "cnt", "residuo"]]
peores["predicho"] = (peores["cnt"] - peores["residuo"]).round()
print(peores.round(0))
plt.figure(figsize=(12, 3)); plt.plot(df["dteday"], df["residuo"]); plt.axhline(0, color="k"); plt.title("Residuos: real menos predicho"); plt.show()
Xq = X.assign(temp_c2=X["temp_c"] ** 2)
m3 = LinearRegression().fit(Xq, y)
b, a = m3.coef_[0], m3.coef_[-1]
print("R² con T²:", round(r2_score(y, m3.predict(Xq)), 3), " b", round(b, 1), " a", round(a, 2), " máximo en", round(-b / (2 * a), 1), "°C")

## 4. Validar en el tiempo

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
pa = LinearRegression().fit(Xtr, ytr).predict(Xte)
print("Al azar:", round(r2_score(yte, pa), 3), round(mean_absolute_error(yte, pa)), len(yte))
tr, te = df["yr"] == 0, df["yr"] == 1
Xs = X.drop(columns="yr")
pt = LinearRegression().fit(Xs[tr], y[tr]).predict(Xs[te])
print("2011 -> 2012 sin año:", round(r2_score(y[te], pt), 3), round(mean_absolute_error(y[te], pt)), " residuo medio", round((y[te] - pt).mean()))
corte = df["dteday"] < "2012-09-01"
pc = LinearRegression().fit(X[corte], y[corte]).predict(X[~corte])
print("Corte septiembre 2012:", round(r2_score(y[~corte], pc), 3), round(mean_absolute_error(y[~corte], pc)), (~corte).sum(), "días")

## 5. La flota del sábado

In [ ]:
fila = pd.DataFrame([{c: 0.0 for c in X.columns}])
fila.loc[0, ["temp_c", "hum_pct", "viento_kmh", "yr", "season_3"]] = [24, 60, 12, 1, 1]
print("Múltiple:", round(m2.predict(fila)[0]), " Con T²:", round(m3.predict(fila.assign(temp_c2=24 ** 2))[0]))

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.